# Migration Plan Generator

This notebook reads assessment data from `MIGRATION_ASSESSMENTS.INTAKE`, builds a structured migration plan, and generates a clean, modern HTML report (print to PDF).

**Usage:** Set `CUSTOMER_NAME` in the next cell (partial match supported), then Run All.

In [ ]:
%%sql -r ctx
USE DATABASE MIGRATION_ASSESSMENTS;
USE SCHEMA INTAKE;

In [ ]:
CUSTOMER_NAME = "TitanForge"

In [ ]:
import json, os, re
print('Using HTML-to-PDF approach (browser print-optimized)')

In [ ]:
from datetime import datetime
from snowflake.snowpark.context import get_active_session

session = get_active_session()

row = session.sql(f"SELECT * FROM MIGRATION_ASSESSMENTS.INTAKE.ASSESSMENTS WHERE UPPER(CUSTOMER_NAME) LIKE UPPER('%{CUSTOMER_NAME}%') ORDER BY SUBMITTED_AT DESC LIMIT 1").collect()
if not row:
    raise ValueError(f'No assessment found matching "{CUSTOMER_NAME}". Run: SELECT CUSTOMER_NAME FROM MIGRATION_ASSESSMENTS.INTAKE.ASSESSMENTS;')
row = row[0].as_dict()

assessment_id = row['ASSESSMENT_ID']
cust = json.loads(row['CUSTOMER_DATA']) if isinstance(row['CUSTOMER_DATA'], str) else row['CUSTOMER_DATA']
vol = json.loads(row['VOLUME_DATA']) if isinstance(row['VOLUME_DATA'], str) else row['VOLUME_DATA']
etl = json.loads(row['ETL_DATA']) if isinstance(row['ETL_DATA'], str) else row['ETL_DATA']
trans = json.loads(row['TRANSFORMATION_DATA']) if isinstance(row['TRANSFORMATION_DATA'], str) else row['TRANSFORMATION_DATA']
nuances = json.loads(row['NUANCES_DATA']) if isinstance(row['NUANCES_DATA'], str) else row['NUANCES_DATA']

src_rows = session.sql(f"SELECT * FROM MIGRATION_ASSESSMENTS.INTAKE.ASSESSMENT_SOURCE_SYSTEMS WHERE ASSESSMENT_ID = '{assessment_id}' ORDER BY DATA_SIZE_GB DESC").collect()
etl_rows = session.sql(f"SELECT * FROM MIGRATION_ASSESSMENTS.INTAKE.ASSESSMENT_ETL_TOOLS WHERE ASSESSMENT_ID = '{assessment_id}' ORDER BY PIPELINE_COUNT DESC").collect()

sources = row['SOURCE_SYSTEMS'] if isinstance(row['SOURCE_SYSTEMS'], list) else json.loads(row['SOURCE_SYSTEMS'])
drivers = row['MIGRATION_DRIVERS'] if isinstance(row['MIGRATION_DRIVERS'], list) else json.loads(row['MIGRATION_DRIVERS'])
compliance = row['COMPLIANCE_REQS'] if isinstance(row['COMPLIANCE_REQS'], list) else json.loads(str(row['COMPLIANCE_REQS'])) if row['COMPLIANCE_REQS'] else []

meta = {
    'customer_name': row['CUSTOMER_NAME'],
    'assessment_id': assessment_id,
    'sa_name': row['SA_NAME'] or '',
    'sa_email': row['SA_EMAIL'] or '',
    'generated_date': datetime.now().strftime('%B %d, %Y'),
    'generated_by': 'kipi.ai CoCoDuo Migration Expert',
}

def safe(d, k, default=''):
    v = d.get(k, default) if d else default
    return str(v) if v else default

sp_total = 0
src_sys_table = []
for sr in src_rows:
    sd = sr.as_dict()
    spec = json.loads(sd['SYSTEM_SPECIFIC_DETAILS']) if isinstance(sd['SYSTEM_SPECIFIC_DETAILS'], str) else (sd['SYSTEM_SPECIFIC_DETAILS'] or {})
    procs = spec.get('plsql_packages', 0) + spec.get('plsql_procedures', 0) + spec.get('stored_procedures', 0)
    sp_total += procs
    cx = []
    for key, label in [('plsql_packages','PL/SQL pkgs'),('plsql_procedures','procs'),('stored_procedures','SPs'),('ssis_packages','SSIS'),('ssrs_reports','SSRS'),('triggers','triggers'),('bteq_scripts','BTEQ'),('notebooks','notebooks'),('delta_tables','Delta tables'),('jobs','jobs')]:
        if spec.get(key): cx.append(f"{spec[key]} {label}")
    src_sys_table.append({'name': sd['SOURCE_SYSTEM'], 'version': sd['VERSION'] or '', 'size': f"{sd['DATA_SIZE_GB']/1000:.0f} TB" if sd['DATA_SIZE_GB'] and sd['DATA_SIZE_GB'] >= 1000 else f"{sd['DATA_SIZE_GB']:.0f} GB", 'tables': f"{sd['TABLE_COUNT']:,}" if sd['TABLE_COUNT'] else '0', 'complexity': ', '.join(cx) if cx else 'Standard'})

tool_table = []
for er in etl_rows:
    ed = er.as_dict()
    tool_table.append({'category': 'ETL', 'tool': ed['TOOL_NAME'], 'pipelines': str(ed['PIPELINE_COUNT'] or '-'), 'satisfaction': ed['SATISFACTION'] or '-', 'action': ed['POST_MIGRATION_PLAN'] or '-'})

pain_points = []
for k in ['tooling_concerns', 'technical_debt', 'cost_priorities']:
    v = nuances.get(k, '') if nuances else ''
    if v: pain_points.append(v[:300])

edition = safe(cust, 'snowflake_edition', 'Enterprise')
budget = safe(cust, 'budget_sensitivity', 'Moderate')
is_cost = 'cost' in budget.lower() or 'open source' in budget.lower()

if safe(nuances, 'data_architecture_vision').lower().find('medallion') >= 0 or safe(nuances, 'data_architecture_vision').lower().find('bronze') >= 0:
    arch_pattern = 'Medallion Architecture (Bronze / Silver / Gold)'
    schemas = [{'name': 'BRONZE', 'purpose': 'Raw ingested data'}, {'name': 'SILVER', 'purpose': 'Cleaned & conformed'}, {'name': 'GOLD', 'purpose': 'Business-ready'}, {'name': 'SANDBOX', 'purpose': 'Exploration'}]
else:
    arch_pattern = 'Layered ELT Architecture'
    schemas = [{'name': 'RAW', 'purpose': 'Raw ingested data'}, {'name': 'TRANSFORM', 'purpose': 'Cleaned & joined'}, {'name': 'ANALYTICS', 'purpose': 'Data marts'}]

wh_strategy = [
    {'name': 'WH_LOADING', 'size': 'X-Small' if is_cost else 'Small', 'auto_suspend': '60s', 'purpose': 'Bulk loading, Snowpipe', 'est_monthly': '~$200' if is_cost else '~$500'},
    {'name': 'WH_TRANSFORM', 'size': 'Small' if is_cost else 'Medium', 'auto_suspend': '120s', 'purpose': 'dbt, SP execution', 'est_monthly': '~$800' if is_cost else '~$2,000'},
    {'name': 'WH_ANALYTICS', 'size': 'Small (MC 1-3)' if is_cost else 'Medium (MC 1-5)', 'auto_suspend': '300s', 'purpose': 'BI queries, dashboards', 'est_monthly': '~$1,200' if is_cost else '~$3,000'},
    {'name': 'WH_DEV', 'size': 'X-Small', 'auto_suspend': '60s', 'purpose': 'Development, CI/CD', 'est_monthly': '~$200'},
]

rec_tools = []
for t in (etl.get('ingestion_preferences', []) if etl else []): rec_tools.append({'category': 'Ingestion', 'tool': t, 'justification': 'Selected by SA', 'est_annual_cost': '-'})
for t in (trans.get('target_transform', []) if trans else []): rec_tools.append({'category': 'Transformation', 'tool': t, 'justification': 'Target per assessment', 'est_annual_cost': '-'})
for t in (trans.get('target_orchestration', []) if trans else []): rec_tools.append({'category': 'Orchestration', 'tool': t, 'justification': 'Target per assessment', 'est_annual_cost': '-'})
for t in (trans.get('target_bi', []) if trans else []): rec_tools.append({'category': 'BI', 'tool': t, 'justification': 'Target per assessment', 'est_annual_cost': '-'})
for t in (trans.get('target_governance', []) if trans else []): rec_tools.append({'category': 'Governance', 'tool': t, 'justification': 'Target per assessment', 'est_annual_cost': '-'})

cost_comp = None
if etl.get('annual_etl_cost') or etl.get('annual_infra_cost'):
    cost_comp = {'current_items': [{'item': 'Infrastructure', 'cost': safe(etl, 'annual_infra_cost', 'N/A')}, {'item': 'ETL', 'cost': safe(etl, 'annual_etl_cost', 'N/A')}], 'future_items': [{'item': 'Snowflake + Tooling', 'cost': 'See warehouse estimates'}], 'total_current': safe(etl, 'annual_infra_cost', '') + ' + ' + safe(etl, 'annual_etl_cost', ''), 'total_future': 'Projected lower', 'savings': 'Substantial', 'savings_pct': 'TBD'}

timeline = safe(cust, 'timeline', '6-12 months')
approach = safe(cust, 'migration_approach', 'Phased')
phases = [{'phase_num': 0, 'title': 'Foundation & Quick Wins', 'weeks': 'Weeks 1-6', 'tasks': ['Snowflake account provisioning', 'Network setup', 'RBAC + warehouses + resource monitors', 'CI/CD pipeline', 'Ingestion tool deployment', 'Quick Win: first table + dashboard on Snowflake']}]
for i, src in enumerate(sources):
    phases.append({'phase_num': i+1, 'title': f'{src} Migration', 'weeks': f'Weeks {7+i*10}-{16+i*10}', 'tasks': [f'Bulk load {src} tables', f'Convert {src} procedures/ETL', f'Build dbt models for {src}', 'CDC setup', 'Validation: counts, checksums, KPIs']})
phases.append({'phase_num': len(sources)+1, 'title': 'Parallel Run & Validation', 'weeks': f'Weeks {7+len(sources)*10}-{16+len(sources)*10}', 'tasks': ['Full parallel run', 'KPI reconciliation', 'Performance benchmarking', 'UAT', 'Load testing']})
phases.append({'phase_num': len(sources)+2, 'title': 'Cutover & Decommission', 'weeks': f'Weeks {17+len(sources)*10}-{24+len(sources)*10}', 'tasks': ['Production cutover', 'Decommission legacy systems', '30-day hypercare', 'Cost optimization pass']})

sec_comp = []
for c in compliance:
    m = {'HIPAA': 'BAA + PHI masking + audit', 'SOC': 'SOC 2 Type II + access controls', 'GDPR': 'Classification + right-to-erasure', 'PCI': 'Column masking + network policies', 'SOX': 'Immutable audit trail', 'CCPA': 'Deletion workflows + consent', 'ISO': 'Standard security controls'}
    matched = False
    for k, v in m.items():
        if k in c: sec_comp.append({'requirement': c, 'implementation': v}); matched = True; break
    if not matched: sec_comp.append({'requirement': c, 'implementation': 'Standard Snowflake security'})
sec_comp += [{'requirement': 'Dynamic Data Masking', 'implementation': 'PII/PHI masking, role-based'}, {'requirement': 'Encryption', 'implementation': 'AES-256 at rest, TLS 1.2+ in transit'}, {'requirement': 'Audit Trail', 'implementation': 'Access History + Account Usage'}]

risks = []
if nuances.get('migration_risks'):
    for line in nuances['migration_risks'].split('.'):
        line = line.strip()
        if len(line) > 20: risks.append({'risk': line[:80], 'severity': 'High', 'mitigation': 'See Architecture Nuances'})
if not risks: risks = [{'risk': 'Timeline slippage', 'severity': 'Medium', 'mitigation': 'Buffer weeks in each phase'}]

plan = {
    'metadata': meta,
    'executive_summary': {
        'overview': f"{meta['customer_name']} migrating from {', '.join(sources)} to Snowflake {edition} on {safe(cust, 'target_cloud', 'AWS')}. {row.get('TOTAL_DATA_SIZE_TB', 0)} TB, {row.get('TOTAL_TABLES', 0):,} tables, {sp_total:,}+ SPs.",
        'drivers': 'Drivers: ' + ', '.join(drivers) + '.',
        'approach': f"{approach} over {timeline}. Budget: {budget}.",
        'key_metrics': [
            {'metric': 'Total Data Volume', 'value': f"{row.get('TOTAL_DATA_SIZE_TB', 0)} TB across {len(sources)} systems"},
            {'metric': 'Total Tables', 'value': f"{row.get('TOTAL_TABLES', 0):,}"},
            {'metric': 'Stored Procedures / Packages', 'value': f"{sp_total:,}+"},
            {'metric': 'Active Users', 'value': f"{row.get('TOTAL_USERS', 0):,}"},
            {'metric': 'Daily Queries', 'value': f"{vol.get('daily_queries', 0):,}"},
            {'metric': 'CDC Tables', 'value': str(etl.get('cdc_tables', 0))},
            {'metric': 'Compliance', 'value': ', '.join(compliance) if compliance else 'N/A'},
            {'metric': 'Timeline', 'value': timeline},
            {'metric': 'Budget Sensitivity', 'value': budget},
        ],
    },
    'current_state': {'source_systems': src_sys_table, 'tool_landscape': tool_table, 'pain_points': pain_points},
    'target_architecture': {
        'account_edition': {'edition': edition, 'justification': f"For {', '.join(compliance)}" if compliance else 'Enterprise features', 'cloud_region': safe(cust, 'target_cloud', 'TBD'), 'account_structure': 'Single account, DEV/STAGING/PROD via CLONE'},
        'data_architecture': {'pattern': arch_pattern, 'description': safe(nuances, 'data_architecture_vision', 'Layered ELT with clear separation.'), 'schemas': schemas},
        'multi_tenancy': {'pattern': 'N/A', 'description': ''},
        'warehouses': wh_strategy,
    },
    'recommended_tools': {'tools': rec_tools, 'cost_comparison': cost_comp},
    'migration_phases': phases,
    'security_compliance': sec_comp,
    'risks': risks,
    'success_criteria': [
        {'kpi': 'Data validation', 'target': '100% row count match', 'measurement': 'Automated reconciliation'},
        {'kpi': 'Query performance', 'target': 'Meet SLA targets', 'measurement': 'Query History P95'},
        {'kpi': 'Cost reduction', 'target': budget, 'measurement': 'Snowflake billing vs legacy'},
        {'kpi': 'User adoption', 'target': f"{row.get('TOTAL_USERS', 0)} users in 30 days", 'measurement': 'Access History'},
        {'kpi': 'Legacy decommission', 'target': 'Within timeline', 'measurement': 'License cancellation'},
    ],
    'next_steps': [f"Kickoff with {meta['sa_name']} and {meta['customer_name']}", 'Snowflake account provisioning', 'Network connectivity setup', 'SnowConvert assessment', 'Deploy ingestion + CI/CD', 'First Quick Win in 6 weeks'],
}
print(f"Plan built for: {meta['customer_name']}")
print(f"Assessment: {assessment_id} | Sources: {', '.join(sources)}")
print(f"Data: {row.get('TOTAL_DATA_SIZE_TB', 0)} TB | {row.get('TOTAL_TABLES', 0):,} tables | {sp_total:,} SPs")

In [ ]:
TOOL_DB = {
    'Airbyte': {'sf_fit':8,'ease':7,'enterprise':5,'cost':9,'tag':'Cost-sensitive, open-source','strengths':['Open-source, no per-row pricing','350+ connectors, growing fast','Self-hosted or cloud options','Extensible SDK for custom connectors'],'watchouts':['Self-hosted requires DevOps','Some connectors less mature','Limited built-in monitoring']},
    'Fivetran': {'sf_fit':9,'ease':9,'enterprise':8,'cost':4,'tag':'Small teams, zero-ops SaaS','strengths':['500+ connectors, gold-standard reliability','Zero-maintenance, fully managed','Auto schema detection','Excellent Snowflake integration'],'watchouts':['Per-MAR pricing punishes volume','Very expensive at scale ($100K-$500K+)','Limited transformation']},
    'Matillion': {'sf_fit':9,'ease':7,'enterprise':7,'cost':6,'tag':'ELT + transform in one tool','strengths':['Snowflake-native pushdown SQL','Visual builder + code editor','Loading + transformation combined','Good CI/CD and Git integration'],'watchouts':['Credit pricing unpredictable','Steeper learning curve','Fewer SaaS connectors']},
    'Informatica Cloud (IDMC)': {'sf_fit':7,'ease':5,'enterprise':9,'cost':3,'tag':'Enterprise legacy, SAP, mainframe','strengths':['Broadest connector library (SAP, mainframe)','Enterprise governance, MDM, DQ','Legacy migration heritage','CLAIRE AI for mapping'],'watchouts':['Very expensive ($300K-$1M+)','Complex UI, steep learning curve','Vendor lock-in, slow innovation']},
    'Hevo Data': {'sf_fit':7,'ease':8,'enterprise':5,'cost':7,'tag':'Mid-market, simple managed','strengths':['Simple UI, fast setup','Transparent per-event pricing','Good CDC support','Responsive support'],'watchouts':['Fewer enterprise connectors','Less mature ecosystem','Limited transformation']},
    'Stitch Data': {'sf_fit':5,'ease':7,'enterprise':3,'cost':6,'tag':'Simple SaaS extraction','strengths':['Simple setup','Low volume pricing'],'watchouts':['Frequent sync failures','Poor error handling','Limited connectors']},
    'AWS Glue': {'sf_fit':5,'ease':4,'enterprise':6,'cost':5,'tag':'AWS-native PySpark ETL','strengths':['Serverless','AWS integration','No infrastructure to manage'],'watchouts':['Slow startup, painful debugging','Unpredictable costs','PySpark overhead for simple SQL']},
    'Azure Data Factory (ADF)': {'sf_fit':6,'ease':7,'enterprise':7,'cost':7,'tag':'Azure-native orchestration','strengths':['Azure integration','Visual pipelines','Managed IR for hybrid'],'watchouts':['Snowflake connector not native','Limited transformation','Activity-based pricing']},
    'Custom Scripts (Python/Spark)': {'sf_fit':5,'ease':4,'enterprise':2,'cost':9,'tag':'Custom code, no licensing','strengths':['Full flexibility','No licensing cost','Any source/target'],'watchouts':['No monitoring or alerting','Fragile, undocumented','Hard to maintain and scale']},
    'SSIS (SQL Server Integration Services)': {'sf_fit':3,'ease':5,'enterprise':6,'cost':5,'tag':'SQL Server native ETL','strengths':['Native SQL Server integration','Mature error handling'],'watchouts':['Windows-only','No Snowflake-native support','Legacy, declining']},
    'Informatica PowerCenter': {'sf_fit':4,'ease':4,'enterprise':8,'cost':2,'tag':'Legacy enterprise ETL','strengths':['Mature','Feature-rich for on-prem'],'watchouts':['End-of-life trajectory','Very expensive','Hard to hire talent']},
    'Ab Initio': {'sf_fit':3,'ease':3,'enterprise':7,'cost':1,'tag':'Legacy parallel ETL','strengths':['High performance parallel processing'],'watchouts':['Extremely expensive ($500K-$1M+)','Impossible to hire talent','Total vendor lock-in']},
    'dbt-core': {'sf_fit':9,'ease':7,'enterprise':5,'cost':10,'tag':'Free, SQL-first transformation','strengths':['Free and open-source','SQL-first for analysts','Excellent testing & docs','Massive community'],'watchouts':['No built-in scheduler','CLI only, no visual UI','Enterprise features need dbt Cloud']},
    'dbt Cloud': {'sf_fit':9,'ease':9,'enterprise':8,'cost':6,'tag':'Managed CI/CD for dbt','strengths':['Built-in scheduler, CI/CD, IDE','Snowflake-optimized','Managed docs & freshness','Slim CI for changed models'],'watchouts':['$100-$500/seat/month','Overkill for solo devs','Some features locked to Enterprise']},
    'dbt (data build tool)': {'sf_fit':9,'ease':7,'enterprise':5,'cost':10,'tag':'Free, SQL-first transformation','strengths':['Free and open-source','SQL-first for analysts','Excellent testing & docs','Massive community'],'watchouts':['No built-in scheduler','CLI only','Enterprise features need dbt Cloud']},
    'Coalesce': {'sf_fit':9,'ease':9,'enterprise':6,'cost':6,'tag':'Visual transformation builder','strengths':['Drag-and-drop visual builder','Snowflake-native SQL generation','Column-level lineage','Good for mixed-skill teams'],'watchouts':['Smaller community than dbt','Snowflake-only','Less Python flexibility']},
    'SQLMesh': {'sf_fit':8,'ease':7,'enterprise':5,'cost':10,'tag':'dbt alternative with environments','strengths':['Built-in virtual environments','Change categorization','dbt compatibility mode','Free and open-source'],'watchouts':['Much smaller community','Less ecosystem support','Newer, less battle-tested']},
    'Snowpark (Python / Java / Scala)': {'sf_fit':10,'ease':6,'enterprise':7,'cost':10,'tag':'Native Snowflake code execution','strengths':['Runs in Snowflake, no data movement','Python/Java/Scala support','ML feature engineering','Integrates with Snowpark ML'],'watchouts':['Steeper learning curve','Debugging challenges','Overkill for simple SQL']},
    'Stored Procedures (in-database)': {'sf_fit':7,'ease':6,'enterprise':5,'cost':10,'tag':'Native database logic','strengths':['No external tool needed','Runs in Snowflake','Good for simple procedural logic'],'watchouts':['Less testable than dbt','No lineage or documentation','Hard to version control']},
    'Tableau': {'sf_fit':8,'ease':7,'enterprise':9,'cost':5,'tag':'Enterprise visualization leader','strengths':['Industry-leading visualizations','Large talent pool','Snowflake Live Query','Enterprise governance'],'watchouts':['Expensive ($70+/user/mo)','Complex for self-service','Extracts create data sprawl']},
    'Sigma Computing': {'sf_fit':10,'ease':9,'enterprise':7,'cost':7,'tag':'Snowflake-native, self-service','strengths':['Built natively for Snowflake','Spreadsheet UX for business users','No extracts, always live','Excellent self-service'],'watchouts':['Smaller ecosystem','Less pixel-perfect reporting','Growing enterprise track record']},
    'Power BI': {'sf_fit':6,'ease':8,'enterprise':8,'cost':8,'tag':'Affordable, Microsoft ecosystem','strengths':['$10/user/mo, very affordable','Strong Microsoft integration','Large community','AI Copilot'],'watchouts':['DirectQuery perf limitations','Import copies data','Less Snowflake-native']},
    'ThoughtSpot': {'sf_fit':9,'ease':9,'enterprise':7,'cost':5,'tag':'AI-driven NLQ analytics','strengths':['Natural language search','SpotIQ AI insights','Snowflake-native','Democratizes data'],'watchouts':['Expensive per-seat','Less custom viz','Needs well-modeled data']},
    'Looker': {'sf_fit':7,'ease':6,'enterprise':7,'cost':5,'tag':'LookML governed metrics','strengths':['LookML for governed metrics','Google ecosystem','Strong modeling layer'],'watchouts':['Steep LookML learning curve','Google acquisition uncertainty','Expensive']},
    'Metabase': {'sf_fit':6,'ease':8,'enterprise':4,'cost':9,'tag':'Free open-source BI','strengths':['Free self-hosted','Simple, good for small teams','Quick dashboard creation'],'watchouts':['Limited enterprise governance','Self-hosted maintenance','Less polished']},
    'Apache Superset': {'sf_fit':6,'ease':6,'enterprise':4,'cost':9,'tag':'Free open-source BI','strengths':['Free','Developer-friendly','SQL-native'],'watchouts':['Complex setup','Limited self-service','Smaller community']},
    'Streamlit': {'sf_fit':10,'ease':7,'enterprise':4,'cost':10,'tag':'Custom data apps, zero cost','strengths':['Zero licensing, runs in Snowflake','Python-native','Great for custom apps/ML demos'],'watchouts':['Not a BI tool','Requires coding','No drag-and-drop dashboards']},
    'Apache Airflow / MWAA': {'sf_fit':8,'ease':5,'enterprise':7,'cost':7,'tag':'Complex multi-system orchestration','strengths':['Most adopted orchestrator','Python-native, extensible','Snowflake Provider','Huge community'],'watchouts':['Steep learning curve','Heavy ops if self-hosted','Complex for non-engineers']},
    'Snowflake Tasks & Streams': {'sf_fit':10,'ease':8,'enterprise':6,'cost':10,'tag':'Zero-cost native scheduling','strengths':['Zero additional cost','Native Snowflake','Streams for CDC','Serverless Tasks'],'watchouts':['Snowflake-only workflows','No visual DAG editor','No multi-system orchestration']},
    'Prefect': {'sf_fit':7,'ease':7,'enterprise':6,'cost':8,'tag':'Modern Python orchestrator','strengths':['Modern DX, better than Airflow','Python-native','Hybrid execution'],'watchouts':['Smaller community','Less battle-tested','Fewer integrations']},
    'Dagster': {'sf_fit':7,'ease':7,'enterprise':6,'cost':8,'tag':'Asset-centric orchestration','strengths':['Asset-centric paradigm','Built-in lineage','dbt integration'],'watchouts':['Smaller ecosystem','Learning curve','Newer']},
    'dbt Cloud Scheduler': {'sf_fit':9,'ease':9,'enterprise':7,'cost':6,'tag':'Built-in dbt scheduling','strengths':['Included with dbt Cloud','Zero setup','Slim CI'],'watchouts':['Only schedules dbt models','Needs dbt Cloud license','No multi-system orchestration']},
    'Cron / Custom Scheduler': {'sf_fit':3,'ease':6,'enterprise':2,'cost':10,'tag':'Basic free scheduling','strengths':['Free','Simple'],'watchouts':['No monitoring','No retry logic','No DAG dependencies']},
    'Control-M': {'sf_fit':5,'ease':4,'enterprise':8,'cost':3,'tag':'Legacy enterprise scheduler','strengths':['Enterprise-grade','Mature'],'watchouts':['Expensive licensing','Legacy architecture','Hard to integrate with cloud']},
    'Snowflake Horizon (native)': {'sf_fit':10,'ease':8,'enterprise':7,'cost':10,'tag':'Native governance, zero cost','strengths':['Included with Snowflake','Masking, RLS, Tagging, Classification','Access History and lineage','Auto PII detection'],'watchouts':['Snowflake-only','Less rich UI than dedicated tools','No cross-platform catalog']},
    'Snowflake Horizon': {'sf_fit':10,'ease':8,'enterprise':7,'cost':10,'tag':'Native governance, zero cost','strengths':['Included with Snowflake','Masking, RLS, Tagging, Classification','Access History and lineage'],'watchouts':['Snowflake-only','Less rich UI']},
    'Monte Carlo': {'sf_fit':9,'ease':8,'enterprise':8,'cost':5,'tag':'ML-powered data observability','strengths':['Automated ML anomaly detection','End-to-end lineage','Incident management','Snowflake-native'],'watchouts':['Expensive per-table','False positives initially','Tuning period']},
    'Atlan': {'sf_fit':8,'ease':9,'enterprise':7,'cost':5,'tag':'Modern collaborative catalog','strengths':['Modern UI, Slack-like','Automated lineage','AI-powered search','Strong Snowflake integration'],'watchouts':['Expensive for small teams','Less mature for compliance']},
    'Great Expectations': {'sf_fit':6,'ease':5,'enterprise':4,'cost':10,'tag':'Open-source Python DQ','strengths':['Free','Python-native','Extensive expectations library'],'watchouts':['Steep learning curve','Requires Python','Operationally heavy']},
    'Soda': {'sf_fit':7,'ease':8,'enterprise':5,'cost':8,'tag':'YAML-based DQ checks','strengths':['Simple YAML checks','dbt + Airflow integration','Open-source core'],'watchouts':['Less automated than Monte Carlo','No ML anomaly detection in free tier']},
    'Snowflake Horizon Catalog (native)': {'sf_fit':10,'ease':8,'enterprise':6,'cost':10,'tag':'Native Snowflake catalog','strengths':['Included with Snowflake','Native tagging and classification'],'watchouts':['Snowflake-only','Less rich than Atlan/Collibra']},
    'DataHub (LinkedIn OSS)': {'sf_fit':6,'ease':5,'enterprise':5,'cost':9,'tag':'Open-source data catalog','strengths':['Free, open-source','Strong lineage','Active community'],'watchouts':['Complex setup','Requires engineering resources']},
}

def get_comparisons():
    comps = []
    tm = cust.get('team_size', 5)
    tooling_notes = safe(nuances, 'tooling_concerns', '').lower()
    dissatisfied_tools = [t['tool'] for t in tool_table if t.get('satisfaction') == 'Dissatisfied']
    satisfied_tools = [t['tool'] for t in tool_table if t.get('satisfaction') == 'Satisfied']

    sa_target_transform = trans.get('target_transform', []) if trans else []
    sa_target_orch = trans.get('target_orchestration', []) if trans else []
    sa_target_bi = trans.get('target_bi', []) if trans else []
    sa_target_gov = trans.get('target_governance', []) if trans else []
    sa_ingestion_prefs = etl.get('ingestion_preferences', []) if etl else []
    sa_current_etl = [t['tool'] for t in tool_table]

    def build_cat(cat, cands, rec_name, rec_reason):
        cards = []
        for n in cands:
            if n in TOOL_DB:
                cards.append({'name': n, **TOOL_DB[n]})
        if not cards: return None
        return {'category': cat, 'cards': cards, 'recommended': rec_name, 'verdict': rec_reason}

    # INGESTION: Use SA's satisfied/dissatisfied tools + ingestion preferences
    etl_cands = []
    etl_rec = None
    etl_reason = ''
    for t in satisfied_tools:
        if t in TOOL_DB: etl_cands.append(t)
    for t in sa_current_etl:
        if t in TOOL_DB and t not in etl_cands: etl_cands.append(t)
    if not etl_cands:
        etl_cands = ['Airbyte', 'Fivetran', 'Matillion']
    etl_cands = etl_cands[:3]
    if satisfied_tools:
        etl_rec = satisfied_tools[0]
        etl_reason = f"SA rated {satisfied_tools[0]} as Satisfied with plan to keep. {'Replacing ' + dissatisfied_tools[0] + ' (Dissatisfied). ' if dissatisfied_tools else ''}" 
    elif dissatisfied_tools:
        alts = [c for c in etl_cands if c not in dissatisfied_tools]
        etl_rec = alts[0] if alts else etl_cands[0]
        etl_reason = f"Replacing {', '.join(dissatisfied_tools)} (SA rated Dissatisfied). {etl_rec} addresses the pain points noted in tooling concerns."
    else:
        etl_rec = etl_cands[0]
        etl_reason = f"Based on {meta['customer_name']}'s profile and SA's ingestion preferences."
    c = build_cat('Data Ingestion / ELT', etl_cands, etl_rec, etl_reason)
    if c: comps.append(c)

    # TRANSFORMATION: Use SA's target_transform directly
    tr_cands = [t for t in sa_target_transform if t in TOOL_DB][:2]
    if len(tr_cands) < 2:
        for fallback in ['dbt Cloud', 'dbt-core', 'Coalesce']:
            if fallback not in tr_cands and fallback in TOOL_DB:
                tr_cands.append(fallback)
                if len(tr_cands) >= 3: break
    tr_rec = tr_cands[0] if tr_cands else 'dbt-core'
    tr_reason = f"SA selected {tr_rec} as the target transformation tool for {meta['customer_name']}."
    if len(sa_target_transform) > 1:
        tr_reason += f" Also evaluating {sa_target_transform[1]}."
    c = build_cat('Data Transformation', tr_cands[:3], tr_rec, tr_reason)
    if c: comps.append(c)

    # BI: Use SA's target_bi directly
    bi_cands = [t for t in sa_target_bi if t in TOOL_DB][:2]
    if len(bi_cands) < 2:
        for fallback in ['Sigma Computing', 'Tableau', 'Power BI']:
            if fallback not in bi_cands and fallback in TOOL_DB:
                bi_cands.append(fallback)
                if len(bi_cands) >= 3: break
    bi_rec = bi_cands[0] if bi_cands else 'Sigma Computing'
    bi_reason = f"SA selected {bi_rec} as the target BI tool."
    c = build_cat('BI & Analytics', bi_cands[:3], bi_rec, bi_reason)
    if c: comps.append(c)

    # GOVERNANCE: Use SA's target_governance
    gov_cands = [t for t in sa_target_gov if t in TOOL_DB][:2]
    if len(gov_cands) < 2:
        for fallback in ['Snowflake Horizon', 'Monte Carlo', 'Atlan']:
            if fallback not in gov_cands and fallback in TOOL_DB:
                gov_cands.append(fallback)
                if len(gov_cands) >= 3: break
    gov_rec = gov_cands[0] if gov_cands else 'Snowflake Horizon'
    gov_reason = f"SA selected {gov_rec} for governance."
    c = build_cat('Governance & Observability', gov_cands[:3], gov_rec, gov_reason)
    if c: comps.append(c)

    # ORCHESTRATION: Use SA's target_orchestration
    or_cands = [t for t in sa_target_orch if t in TOOL_DB][:2]
    if len(or_cands) < 2:
        for fallback in ['Snowflake Tasks & Streams', 'Apache Airflow / MWAA']:
            if fallback not in or_cands and fallback in TOOL_DB:
                or_cands.append(fallback)
                if len(or_cands) >= 2: break
    or_rec = or_cands[0] if or_cands else 'Snowflake Tasks & Streams'
    or_reason = f"SA selected {or_rec} as target orchestrator for {meta['customer_name']}."
    c = build_cat('Orchestration', or_cands[:3], or_rec, or_reason)
    if c: comps.append(c)

    return comps

plan['tool_comparisons'] = get_comparisons()
for tc_item in plan['tool_comparisons']:
    print(f"{tc_item['category']}: {' vs '.join(c['name'] for c in tc_item['cards'])} -> Recommended: {tc_item['recommended']}")

In [ ]:
CSS = """
@page { size: A4; margin: 18mm 16mm; }
@media print {
  body { -webkit-print-color-adjust: exact !important; print-color-adjust: exact !important; }
  .page-break { page-break-before: always; }
  .no-break { page-break-inside: avoid; }
}
:root {
  --g: #B3D943; --gl: #E8F5C8; --gd: #4A5A2B;
  --bk: #1A1A1A; --nv: #2C3E6B; --cy: #29B5E8; --lb: #7ECCE5;
  --dg: #333; --mg: #666; --lg: #F5F5F5; --bg: #E0E0E0; --w: #FFF;
  --rc: #D32F2F; --rh: #F57C00; --rm: #666; --rs: #388E3C;
}
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Proxima Nova','Helvetica Neue',Helvetica,Arial,sans-serif; color: var(--dg); font-size: 11pt; line-height: 1.6; max-width: 210mm; margin: 0 auto; }
.cover { display:flex; flex-direction:column; align-items:center; justify-content:center; min-height:92vh; text-align:center; }
.cover .brand { font-size:13pt; color:var(--g); font-weight:600; letter-spacing:2px; }
.cover .divider { width:60px; height:2px; background:var(--g); margin:20px auto; }
.cover h1 { font-size:32pt; color:var(--bk); font-weight:700; line-height:1.15; margin:4px 0; }
.cover .customer { font-size:17pt; color:var(--mg); font-weight:400; margin:4px 0 16px; }
.cover .meta-table td { padding:4px 12px; font-size:11pt; }
.cover .meta-table td:first-child { color:var(--mg); text-align:right; font-weight:400; }
.cover .meta-table td:last-child { color:var(--bk); font-weight:600; text-align:left; }
.cover .confidential { font-size:9pt; color:#999; font-style:italic; margin-top:40px; }
h2 { font-size:16pt; color:var(--bk); font-weight:700; margin:28px 0 12px; padding-bottom:4px; border-bottom:2px solid var(--g); display:inline-block; }
h3 { font-size:12pt; color:var(--bk); font-weight:600; margin:16px 0 8px; }
.sec { margin-bottom:20px; }
.metrics { display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin:12px 0 16px; }
.metric-card { background:var(--lg); border-radius:6px; padding:14px 12px; text-align:center; border-top:3px solid var(--g); }
.metric-card .val { font-size:22pt; font-weight:700; color:var(--bk); line-height:1.2; }
.metric-card .lbl { font-size:8.5pt; color:var(--mg); letter-spacing:0.5px; margin-top:4px; }
.tag { display:inline-block; padding:3px 10px; border-radius:4px; font-size:9pt; font-weight:600; margin:2px 3px; }
.tag-green { background:var(--gl); color:var(--gd); }
.tag-blue { background:#E3F2FD; color:var(--nv); }
.tag-gray { background:var(--lg); color:var(--dg); }
table { width:100%; border-collapse:collapse; margin:8px 0 14px; font-size:10pt; }
th { background:var(--bk); color:var(--w); padding:8px 10px; text-align:left; font-weight:600; font-size:9pt; letter-spacing:0.3px; }
td { padding:7px 10px; border-bottom:1px solid var(--bg); }
tr:nth-child(even) td { background:var(--lg); }
.timeline { position:relative; margin:14px 0 14px 24px; padding-left:24px; border-left:2px solid var(--g); }
.tl-item { position:relative; margin-bottom:18px; }
.tl-item::before { content:''; position:absolute; left:-30px; top:4px; width:12px; height:12px; border-radius:50%; background:var(--g); border:2px solid var(--w); }
.tl-item .tl-head { font-size:11pt; font-weight:700; color:var(--bk); }
.tl-item .tl-sub { font-size:9pt; color:var(--mg); margin-bottom:4px; }
.tl-item ul { margin:4px 0 0 16px; font-size:10pt; }
.tl-item li { margin-bottom:3px; }
.risk-grid { display:grid; grid-template-columns:1fr 1fr; gap:8px; margin:10px 0; }
.risk-card { padding:10px 12px; border-radius:5px; border-left:4px solid; font-size:10pt; }
.risk-card.critical { border-color:var(--rc); background:#FFF5F5; }
.risk-card.high { border-color:var(--rh); background:#FFF8F0; }
.risk-card.medium { border-color:var(--mg); background:var(--lg); }
.risk-card .risk-title { font-weight:700; font-size:10pt; }
.risk-card .risk-mit { color:var(--mg); font-size:9pt; margin-top:3px; }
.kpi-row { display:flex; align-items:center; padding:8px 0; border-bottom:1px solid var(--bg); font-size:10pt; }
.kpi-row .kpi-name { width:35%; font-weight:600; color:var(--bk); }
.kpi-row .kpi-target { width:35%; color:var(--dg); }
.kpi-row .kpi-method { width:30%; color:var(--mg); font-size:9pt; }
.schema-flow { display:flex; gap:8px; margin:10px 0; flex-wrap:wrap; }
.schema-box { flex:1; min-width:90px; background:var(--lg); border-top:3px solid var(--g); border-radius:5px; padding:10px; text-align:center; }
.schema-box .s-name { font-size:10pt; font-weight:700; color:var(--bk); }
.schema-box .s-desc { font-size:8pt; color:var(--mg); margin-top:3px; }
.schema-arrow { display:flex; align-items:center; color:var(--g); font-size:16pt; font-weight:700; }
.comp-grid { display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin:10px 0; }
.comp-card { background:var(--lg); border-radius:5px; padding:10px 12px; }
.comp-card .comp-req { font-size:10pt; font-weight:700; color:var(--bk); }
.comp-card .comp-impl { font-size:8.5pt; color:var(--mg); margin-top:3px; }
.steps ol { counter-reset:step; list-style:none; margin:10px 0; padding:0; }
.steps li { counter-increment:step; display:flex; align-items:flex-start; margin-bottom:8px; font-size:10.5pt; }
.steps li::before { content:counter(step); background:var(--g); color:var(--w); font-weight:700; font-size:9pt; width:24px; height:24px; border-radius:50%; display:flex; align-items:center; justify-content:center; margin-right:10px; flex-shrink:0; }
.tc-cards { display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:10px; margin:10px 0; }
.tc-card { background:var(--w); border:1px solid var(--bg); border-radius:6px; padding:14px; }
.tc-card.recommended { border:2px solid var(--g); }
.tc-card .tc-name { font-size:12pt; font-weight:700; color:var(--bk); }
.tc-card .tc-tag { display:inline-block; font-size:7.5pt; font-weight:600; padding:2px 6px; border-radius:3px; background:var(--gl); color:var(--gd); margin:3px 0 8px; }
.tc-card.recommended .tc-tag { background:var(--g); color:var(--w); }
.rating { display:flex; align-items:center; margin:3px 0; font-size:8.5pt; }
.rating .r-label { width:55%; color:var(--mg); }
.rating .r-bar-bg { width:30%; height:6px; background:var(--bg); border-radius:3px; overflow:hidden; }
.rating .r-bar { height:100%; border-radius:3px; }
.rating .r-score { width:15%; text-align:right; font-weight:600; color:var(--bk); font-size:9pt; }
.sw-section { margin-top:10px; }
.sw-item { display:flex; align-items:flex-start; font-size:8.5pt; margin:2px 0; line-height:1.4; }
.sw-item .sw-dot { width:6px; height:6px; border-radius:50%; margin:4px 6px 0 0; flex-shrink:0; }
.sw-item .sw-dot.green { background:var(--rs); }
.sw-item .sw-dot.red { background:var(--rc); }
.verdict { background:var(--gl); border-left:3px solid var(--g); padding:10px 14px; margin:10px 0; border-radius:0 5px 5px 0; }
.verdict .v-label { font-size:8pt; font-weight:700; color:var(--gd); letter-spacing:0.5px; margin-bottom:2px; }
.verdict .v-text { font-size:9.5pt; color:var(--bk); }
p { margin:4px 0 8px; font-size:10.5pt; }
ul { margin:4px 0 10px 18px; } li { margin-bottom:3px; font-size:10pt; }
.footer-note { font-size:8.5pt; color:#999; font-style:italic; margin-top:28px; padding-top:10px; border-top:1px solid var(--bg); }
"""
print('CSS ready — kipi.ai brand v7 (Proxima Nova + #B3D943 + tool comparison cards).')

In [ ]:
def tbl(headers, rows):
    h = ''.join(f'<th>{c}</th>' for c in headers)
    b = ''.join('<tr>' + ''.join(f'<td>{c}</td>' for c in r) + '</tr>' for r in rows)
    return f'<table><thead><tr>{h}</tr></thead><tbody>{b}</tbody></table>'

def metric_card(val, label):
    return f'<div class="metric-card no-break"><div class="val">{val}</div><div class="lbl">{label}</div></div>'

def tag(text, cls='green'):
    return f'<span class="tag tag-{cls}">{text}</span>'

def risk_card(risk, sev, mit):
    cls = {'CRITICAL':'critical','High':'high'}.get(sev,'medium')
    return f'<div class="risk-card {cls} no-break"><div class="risk-title">{sev}: {risk}</div><div class="risk-mit">{mit}</div></div>'

def rating_bar(label, score, color='#B3D943'):
    pct = score * 10
    return f'<div class="rating"><span class="r-label">{label}</span><div class="r-bar-bg"><div class="r-bar" style="width:{pct}%;background:{color}"></div></div><span class="r-score">{score}/10</span></div>'

def tool_comp_card(t, is_rec=False):
    rec_cls = ' recommended' if is_rec else ''
    rec_badge = ' &#10004; Recommended' if is_rec else ''
    h = f'<div class="tc-card{rec_cls} no-break"><div class="tc-name">{t["name"]}{rec_badge}</div>'
    h += f'<div class="tc-tag">{t["tag"]}</div>'
    colors = {'sf_fit':'#29B5E8','ease':'#B3D943','enterprise':'#F57C00','cost':'#2C3E6B'}
    labels = {'sf_fit':'Snowflake Fit','ease':'Ease of Use','enterprise':'Enterprise Depth','cost':'Total Cost (10=cheapest)'}
    for k in ['sf_fit','ease','enterprise','cost']:
        h += rating_bar(labels[k], t[k], colors[k])
    h += '<div class="sw-section">'
    for s in t.get('strengths',[])[:4]:
        h += f'<div class="sw-item"><div class="sw-dot green"></div><span>{s}</span></div>'
    for w in t.get('watchouts',[])[:3]:
        h += f'<div class="sw-item"><div class="sw-dot red"></div><span>{w}</span></div>'
    h += '</div></div>'
    return h

es = plan['executive_summary']
cs = plan['current_state']
ta = plan['target_architecture']
rt = plan['recommended_tools']
sc = plan['security_compliance']
tc_data = plan.get('tool_comparisons', [])

sa_line = meta['sa_name']
if meta.get('sa_email'): sa_line += f" | {meta['sa_email']}"
src_tags = ' '.join(tag(s['name'], 'blue') for s in cs['source_systems'])
comp_tags = ' '.join(tag(c, 'gray') for c in (compliance if compliance else ['N/A']))
driver_tags = ' '.join(tag(d, 'green') for d in drivers)
km = {m['metric']: m['value'] for m in es['key_metrics']}

html = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Migration Plan - {meta['customer_name']}</title><style>{CSS}</style></head><body>
<div class="cover"><div class="brand">kipi.ai &bull; CoCoDuo</div><div class="divider"></div>
<h1>Migration Plan</h1><div class="customer">{meta['customer_name']}</div><div class="divider"></div>
<table class="meta-table"><tbody><tr><td>Solutions Architect</td><td>{sa_line}</td></tr>
<tr><td>Date</td><td>{meta['generated_date']}</td></tr>
<tr><td>Assessment</td><td>{meta['assessment_id']}</td></tr></tbody></table>
<div class="confidential">Confidential &mdash; Prepared by kipi.ai for {meta['customer_name']}</div></div>
<div class="page-break"></div>
<div class="sec"><h2>Snapshot</h2>
<div class="metrics">{metric_card(km.get('Total Data Volume','N/A').split(' ')[0],'Total Data (TB)')}{metric_card(km.get('Total Tables','0'),'Tables')}{metric_card(km.get('Stored Procedures / Packages','0'),'SPs / Packages')}{metric_card(km.get('Active Users','0'),'Users')}</div>
<div class="metrics">{metric_card(km.get('Daily Queries','0'),'Daily Queries')}{metric_card(km.get('CDC Tables','0'),'CDC Tables')}{metric_card(km.get('Timeline','N/A'),'Timeline')}{metric_card(km.get('Budget Sensitivity','N/A'),'Budget')}</div>
<p><strong>Sources:</strong> {src_tags}</p><p><strong>Compliance:</strong> {comp_tags}</p><p><strong>Drivers:</strong> {driver_tags}</p></div>
<div class="page-break"></div>
<div class="sec"><h2>Current State</h2><h3>Source Systems</h3>
{tbl(['System','Version','Size','Tables','Complexity'], [[s['name'],s['version'],s['size'],s['tables'],s['complexity']] for s in cs['source_systems']])}
<h3>ETL Tools</h3>{tbl(['Tool','Pipelines','Satisfaction','Action'], [[t['tool'],t['pipelines'],t['satisfaction'],t['action']] for t in cs['tool_landscape']])}"""
if cs.get('pain_points'):
    html += '<h3>Key Pain Points</h3><ul>' + ''.join(f'<li>{p[:180]}</li>' for p in cs['pain_points']) + '</ul>'
html += '</div><div class="page-break"></div>'
html += f'<div class="sec"><h2>Target Architecture</h2><p>{tag(ta["account_edition"]["edition"],"green")} on {tag(ta["account_edition"]["cloud_region"],"blue")} &mdash; {ta["account_edition"]["account_structure"]}</p>'
html += f'<h3>{ta["data_architecture"]["pattern"]}</h3><div class="schema-flow">'
for i, s in enumerate(ta['data_architecture'].get('schemas', [])):
    if i > 0: html += '<div class="schema-arrow">&rarr;</div>'
    html += f'<div class="schema-box"><div class="s-name">{s["name"]}</div><div class="s-desc">{s["purpose"]}</div></div>'
html += '</div>'
if ta.get('multi_tenancy') and ta['multi_tenancy']['pattern'] != 'N/A':
    html += f'<p><strong>Multi-Tenancy:</strong> {tag(ta["multi_tenancy"]["pattern"],"blue")}</p>'
html += '<h3>Warehouse Strategy</h3>' + tbl(['Warehouse','Size','Suspend','Purpose','$/mo'], [[w['name'],w['size'],w['auto_suspend'],w['purpose'],w['est_monthly']] for w in ta['warehouses']]) + '</div><div class="page-break"></div>'
for section in tc_data:
    html += f'<div class="sec tc-section"><h2>{section["category"]}</h2><div class="tc-cards">'
    for card in section['cards']:
        html += tool_comp_card(card, is_rec=(card['name'] == section['recommended']))
    html += f'</div><div class="verdict"><div class="v-label">Our Recommendation for {meta["customer_name"]}</div><div class="v-text"><strong>{section["recommended"]}</strong> &mdash; {section["verdict"]}</div></div></div><div class="page-break"></div>'
if rt.get('cost_comparison'):
    cc = rt['cost_comparison']
    html += '<div class="sec"><h3>Cost Comparison</h3>' + tbl(['Item','Current','Future'], [[i['item'],i['cost'],''] for i in cc['current_items']] + [[i['item'],'',i['cost']] for i in cc['future_items']]) + '</div>'
html += '<div class="page-break"></div><div class="sec"><h2>Migration Timeline</h2><div class="timeline">'
for ph in plan['migration_phases']:
    html += f'<div class="tl-item no-break"><div class="tl-head">Phase {ph["phase_num"]}: {ph["title"]}</div><div class="tl-sub">{ph["weeks"]}</div><ul>'
    for task in ph['tasks'][:5]: html += f'<li>{task}</li>'
    if len(ph['tasks']) > 5: html += f'<li>+{len(ph["tasks"])-5} more</li>'
    html += '</ul></div>'
html += '</div></div><div class="page-break"></div>'
html += '<div class="sec"><h2>Security &amp; Compliance</h2><div class="comp-grid">'
for s in sc: html += f'<div class="comp-card no-break"><div class="comp-req">{s["requirement"]}</div><div class="comp-impl">{s["implementation"][:100]}</div></div>'
html += '</div></div>'
html += '<div class="sec"><h2>Risks</h2><div class="risk-grid">'
for r in plan['risks']: html += risk_card(r['risk'][:80], r['severity'], r['mitigation'][:120])
html += '</div></div><div class="page-break"></div>'
html += '<div class="sec"><h2>Success Criteria</h2>'
for s in plan['success_criteria']: html += f'<div class="kpi-row"><div class="kpi-name">{s["kpi"]}</div><div class="kpi-target">{s["target"]}</div><div class="kpi-method">{s["measurement"]}</div></div>'
html += '</div><div class="sec steps"><h2>Next Steps</h2><ol>'
for ns in plan['next_steps']: html += f'<li><span>{ns}</span></li>'
html += f'</ol></div><div class="footer-note">Generated by {meta["generated_by"]} | {meta["assessment_id"]} | {meta["generated_date"]}</div></body></html>'

safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', meta['customer_name'])
tmp_path = f'/tmp/{safe_name}_Migration_Plan.html'
with open(tmp_path, 'w') as f: f.write(html)
session = get_active_session()
session.sql("CREATE STAGE IF NOT EXISTS MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') DIRECTORY = (ENABLE = TRUE)").collect()
session.file.put(tmp_path, '@MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS', auto_compress=False, overwrite=True)
print(f'Plan generated: {os.path.getsize(tmp_path)/1024:.1f} KB')
print(f"  SELECT GET_PRESIGNED_URL(@MIGRATION_ASSESSMENTS.INTAKE.MIGRATION_PLANS, '{safe_name}_Migration_Plan.html', 3600);")
from IPython.display import HTML as IPyHTML, display
display(IPyHTML('<div style="padding:10px;background:#E8F5C8;border-left:3px solid #B3D943;border-radius:4px;font-size:13px;"><strong>Done.</strong> Run the presigned URL query above to view. Then Ctrl+P to save as PDF.</div>'))